In [7]:
from __future__ import annotations

import os
import re
import json
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# User configuration
# ============================================================

root_dir = Path(os.getcwd())

dataset_names = [
    "Replogle_K562_essential",
    # "Replogle_RPE",
    # "NormanWeissman2019",
    # "ChangYe",
    # "ZhaoSims2021",
]

groups = ["single", "dual", "multi"]
# groups = ["single"]

pairing_root = Path("/ibex/project/c2366/Perturb_data/")

manifest_dataset_names = {
    "NormanWeissman2019": "Norman_data",
    "Replogle_K562_essential": "Replogle_k562_data",
    "Replogle_RPE": "Replogle_rpe_data",
    "ChangYe": "ChangYe",
    "ZhaoSims2021": "ZhaoSims2021",
}

evaluation_root = Path("/ibex/user/chenj0i/Perturbation/evaluation")

template_config_path = root_dir / "gene_program_config_shared_template.json"
resolved_config_path = root_dir / "gene_program_config_replogle_k562.json"


In [8]:
# Whether to use selected_variants_TEMPLATE_EDIT_ME.csv to keep only selected variants.
USE_SELECTED_VARIANTS = False

# If USE_SELECTED_VARIANTS=True, whether to also include S0 naive mean control.
INCLUDE_S0_WHEN_USING_SELECTED_VARIANTS = True

# If True, missing h5ad or pseudo-control files are skipped during resolution.
ONLY_EXISTING_PROCESSED_FILES = True
REQUIRE_EXISTING_PSEUDO_FILES = True

# If True, writes pseudo_file_index_from_manifest.csv under each dataset/group output_dir.
WRITE_PSEUDO_FILE_INDEX = True


# ============================================================
# Load template config or build fallback config
# ============================================================

if template_config_path.exists():
    with open(template_config_path, "r") as f:
        config = json.load(f)
    print(f"[Loaded template] {template_config_path}")
else:
    print(f"[Warning] Template config not found: {template_config_path}")
    print("[Info] Building a minimal fallback config.")

    config = {
        "global": {
            "layer": None,
            "perturbation_key": "perturbation_label",
            "control_label_values": ["control", "ctrl", "non-targeting", "NT", "NegCtrl"],
            "max_genes": 10000,
            "min_mean": 0.01,
            "min_frac": 0.01,
            "zscore_eps": 1e-6,
            "program_build_groupby": None,
            "max_control_cells_for_programs": 50000,
            "corr_threshold": 0.5,
            "min_correlated_partners": 5,
            "cluster_method": "auto",
            "leiden_resolution": 1.0,
            "min_program_size": 5,
            "max_program_size": None,
            "min_gene_centroid_corr": 0.2,
            "save_corr_matrix": False,
            "pseudo_glob": "**/pseudo_control*.h5ad",
            "chunk_size": 30000,
            "seed": 0,
            "save_cell_level_scores": False,
        },
        "dataset_overrides": {},
    }


# ============================================================
# Update shared-path settings from this notebook cell
# ============================================================

global_cfg = config.setdefault("global", {})

global_cfg.update(
    {
        "path_mode": "shared_pseudo_pairing_pipeline",

        # Pairing data root:
        # /ibex/project/c2366/Perturb_data/<manifest_dataset_name>/<dataset_id>
        "pairing_root": str(pairing_root),

        # Evaluation root:
        # /ibex/user/chenj0i/Perturbation/evaluation/<dataset_id>_pseudo_pairing_evaluation/<group>
        "evaluation_root": str(evaluation_root),

        # Dataset/group selection controlled directly by this notebook cell.
        "dataset_ids": dataset_names,
        "perturbed_groups": groups,

        # Intermediate folder layer under pairing_root.
        "manifest_dataset_names": manifest_dataset_names,

        # Manifest/pseudo-control behavior.
        "use_manifest_pseudo_files": True,
        "pseudo_glob": global_cfg.get("pseudo_glob", "**/pseudo_control*.h5ad"),
        "use_selected_variants": USE_SELECTED_VARIANTS,
        "include_s0_when_using_selected_variants": INCLUDE_S0_WHEN_USING_SELECTED_VARIANTS,
        "selection_table_name": "selected_variants_TEMPLATE_EDIT_ME.csv",
        "only_existing_processed_files": ONLY_EXISTING_PROCESSED_FILES,
        "require_existing_pseudo_files": REQUIRE_EXISTING_PSEUDO_FILES,
        "skip_datasets_without_pseudo": True,
        "write_pseudo_file_index": WRITE_PSEUDO_FILE_INDEX,
    }
)


# ============================================================
# Resolve shared paths
# ============================================================

from gene_program_pipeline_pkg.shared_paths import resolve_shared_config

resolved_config = resolve_shared_config(
    config,
    only_datasets=dataset_names,
    prepare_only=True,
)


# ============================================================
# Save resolved JSON
# ============================================================

# with open(resolved_config_path, "w") as f:
#     json.dump(resolved_config, f, indent=2)

# print(f"[Saved resolved config] {resolved_config_path}")
def make_json_safe(obj):
    """Recursively convert NaN/NaT/pandas NA/numpy scalars/Path objects into JSON-safe values."""
    if obj is None:
        return None

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)

    if isinstance(obj, float):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return obj

    if pd.isna(obj):
        return None

    return obj


resolved_config_safe = make_json_safe(resolved_config)

with open(resolved_config_path, "w") as f:
    json.dump(
        resolved_config_safe,
        f,
        indent=2,
        allow_nan=False,
    )

print(f"[Saved resolved config] {resolved_config_path}")

# ============================================================
# Show resolution summary
# ============================================================

resolved_rows = []
for d in resolved_config.get("datasets", []):
    resolved_rows.append(
        {
            "dataset_id": d.get("dataset_id"),
            "source_dataset_id": d.get("source_dataset_id"),
            "group": d.get("perturbed_group"),
            "control_h5ad": d.get("control_h5ad"),
            "perturbed_h5ad": d.get("perturbed_h5ad"),
            "pairing_dataset_dir": d.get("pairing_dataset_dir"),
            "manifest_path": d.get("manifest_path"),
            "output_dir": d.get("output_dir"),
            "n_pseudo_files": len(d.get("pseudo_files", [])),
        }
    )

resolved_df = pd.DataFrame(resolved_rows)

print(f"[Resolved datasets/groups] {resolved_df.shape[0]}")
display(resolved_df)

skipped = resolved_config.get("shared_path_resolution", {}).get("skipped", [])
if skipped:
    print(f"[Skipped datasets/groups] {len(skipped)}")
    display(pd.DataFrame(skipped))
else:
    print("[Skipped datasets/groups] 0")

[Loaded template] /ibex/user/chenj0i/Perturbation/SEACells/Junfan_scripts/gene_program_shared_full_pipeline/gene_program_config_shared_template.json
[Saved resolved config] /ibex/user/chenj0i/Perturbation/SEACells/Junfan_scripts/gene_program_shared_full_pipeline/gene_program_config_replogle_k562.json
[Resolved datasets/groups] 1


,dataset_id,source_dataset_id,group,control_h5ad,perturbed_h5ad,pairing_dataset_dir,manifest_path,output_dir,n_pseudo_files
0,Replogle_K562_essential_single,Replogle_K562_essential,single,/ibex/user/chenj0i/Perturbation/data/processed...,/ibex/user/chenj0i/Perturbation/data/processed...,/ibex/project/c2366/Perturb_data/Replogle_k562...,/ibex/project/c2366/Perturb_data/Replogle_k562...,/ibex/user/chenj0i/Perturbation/evaluation/Rep...,186


[Skipped datasets/groups] 2


,dataset_id,group,reason,control_h5ad,perturbed_h5ad
0,Replogle_K562_essential,dual,missing control or perturbed h5ad,/ibex/user/chenj0i/Perturbation/data/processed...,/ibex/user/chenj0i/Perturbation/data/processed...
1,Replogle_K562_essential,multi,missing control or perturbed h5ad,/ibex/user/chenj0i/Perturbation/data/processed...,/ibex/user/chenj0i/Perturbation/data/processed...
